# 03 — Forecasting Model Selection and Evaluation
This notebook selects one frozen univariate forecasting specification using development-only expanding-window backtesting. The sealed 1939 holdout is never opened.

## 1. Model-Selection Context and Final-Holdout Boundary
Only the authenticated 1920-01 through 1938-12 development projection is authorized. Holdout integrity metadata may be authenticated, but its target values remain unavailable.

In [17]:
import json
from pathlib import Path
import pandas as pd
from scripts.project_context import get_project_context
from scripts.forecasting_preparation import load_and_validate_forecasting_preparation_handoff
from scripts.forecasting_model_selection import (
    build_artifacts, frozen_specification_catalog, run_all_backtests, select_winner,
    write_forecasting_model_selection_artifacts,
    load_and_validate_forecasting_model_selection_handoff,
)
PROJECT = get_project_context()
PREPARATION_PATH = 'artifacts/preparation/nottem/preparation-handoff.json'
MODEL_SELECTION_PATH = 'artifacts/model-selection/nottem/model-selection-handoff.json'


## 2. Independent Forecasting Preparation-Handoff Loading

In [18]:
preparation = load_and_validate_forecasting_preparation_handoff(
    project_root=PROJECT.root, preparation_handoff_path=PREPARATION_PATH, expected_dataset_slug='nottem'
)
assert len(preparation.development) == 228 and not hasattr(preparation, 'final_holdout')
preparation_payload = json.loads((PROJECT.root / PREPARATION_PATH).read_text(encoding='utf-8'))
input_gate_summary = {'schema': preparation_payload['schema_version'], 'development_rows': len(preparation.development), 'holdout_values_exposed': False}
input_gate_summary


{'schema': 'forecasting-preparation-handoff.v1',
 'development_rows': 228,
 'holdout_values_exposed': False}

## 3. Frozen Development, Backtesting, and Metric Contracts

In [19]:
assert preparation.prediction_contract['problem_type'] == 'time_series_forecasting'
assert preparation.prediction_contract['forecasting_mode'] == 'univariate'
assert preparation.evaluation_contract['primary_metric'] == 'mae'
assert preparation.readiness['model_selection_ready'] is True
schedule = pd.DataFrame(preparation.backtesting_contract['schedule'])
schedule


,fold,train_start,train_end_forecast_origin,training_observations,complete_training_cycles,validation_start,validation_end,validation_observations,seasonal_mase_scale_from_training
0,1,1920-01,1929-12,120,10,1930-01,1930-12,12,2.858333
1,2,1920-01,1930-12,132,11,1931-01,1931-12,12,2.844167
2,3,1920-01,1931-12,144,12,1932-01,1932-12,12,2.805303
3,4,1920-01,1932-12,156,13,1933-01,1933-12,12,2.753472
4,5,1920-01,1933-12,168,14,1934-01,1934-12,12,2.807692
5,6,1920-01,1934-12,180,15,1935-01,1935-12,12,2.787500
6,7,1920-01,1935-12,192,16,1936-01,1936-12,12,2.790000
7,8,1920-01,1936-12,204,17,1937-01,1937-12,12,2.802604
8,9,1920-01,1937-12,216,18,1938-01,1938-12,12,2.782843


## 4. Frozen Baseline and Candidate Specification Catalog
The complete two-baseline plus eight-learned catalog is declared before any forecast is scored.

In [20]:
catalog = frozen_specification_catalog()
catalog_table = pd.DataFrame([{'candidate_id':s['candidate_id'],'role':s['role'],'family':s['family'],'complexity_rank':s['complexity_rank'],'fixed_hyperparameters':s['fixed_hyperparameters'],'multi_step_strategy':s['multi_step_strategy']} for s in catalog])
catalog_table


,candidate_id,role,family,complexity_rank,fixed_hyperparameters,multi_step_strategy
0,seasonal_naive_12,primary_baseline,SeasonalNaive,0,{},direct_known_history_lookup_12_steps
1,naive_last_value,secondary_baseline,NaiveLastValue,1,{},constant_last_training_value_12_steps
2,seasonal_trend_ols,candidate,DeterministicSeasonalTrendOLS,2,"{'intercept': True, 'linear_time_trend': True,...",direct_known_calendar_design_12_steps
3,holt_winters_additive_no_trend,candidate,ExponentialSmoothing,3,"{'trend': None, 'damped_trend': False, 'season...",native_forecast_12
4,holt_winters_additive_damped_trend,candidate,ExponentialSmoothing,4,"{'trend': 'add', 'damped_trend': True, 'season...",native_forecast_12
5,holt_winters_additive_trend,candidate,ExponentialSmoothing,5,"{'trend': 'add', 'damped_trend': False, 'seaso...",native_forecast_12
6,autoreg_lag_1_12_ct,candidate,AutoReg,6,"{'lags': [1, 12], 'trend': 'ct', 'seasonal': F...",recursive_12_step
7,autoreg_lag_1_2_12_ct,candidate,AutoReg,7,"{'lags': [1, 2, 12], 'trend': 'ct', 'seasonal'...",recursive_12_step
8,sarima_100_100_12,candidate,SARIMAX,8,"{'order': [1, 0, 0], 'seasonal_order': [1, 0, ...",native_state_space_12_step
9,sarima_100_011_12,candidate,SARIMAX,9,"{'order': [1, 0, 0], 'seasonal_order': [0, 1, ...",native_state_space_12_step


## 5. Seasonal-Naive and Last-Value Baseline Backtesting
Both baselines are evaluated with one simultaneous 12-step decision at each authenticated origin.

In [21]:
forecast_rows, summaries, fold_audits = run_all_backtests(preparation.development, preparation.backtesting_contract)
summary_table = pd.DataFrame(summaries)
summary_table[summary_table.role.str.endswith('baseline')][['candidate_id','pooled_mae','pooled_rmse','pooled_seasonal_mase','fold_mae_std','long_horizon_mae_h7_h12']]


,candidate_id,pooled_mae,pooled_rmse,pooled_seasonal_mase,fold_mae_std,long_horizon_mae_h7_h12
0,seasonal_naive_12,2.637963,3.338898,0.941686,0.389743,2.557407
1,naive_last_value,10.563889,13.267843,3.770738,2.181979,13.283333


## 6. Learned Candidate Backtesting
Every learned model is rebuilt from scratch inside each fold. Partial candidates remain auditable but are ineligible.

In [22]:
summary_table[summary_table.role == 'candidate'][['candidate_id','complete','eligible','folds_completed','forecast_rows','warnings','failures']]


,candidate_id,complete,eligible,folds_completed,forecast_rows,warnings,failures
2,seasonal_trend_ols,True,True,9,108,[],[]
3,holt_winters_additive_no_trend,True,True,9,108,[],[]
4,holt_winters_additive_damped_trend,True,True,9,108,[],[]
5,holt_winters_additive_trend,True,True,9,108,[],[]
6,autoreg_lag_1_12_ct,True,True,9,108,[],[]
7,autoreg_lag_1_2_12_ct,True,True,9,108,[],[]
8,sarima_100_100_12,False,False,2,24,[DeprecationWarning: Setting the shape on a Nu...,[ForecastingModelSelectionError: Explicit opti...
9,sarima_100_011_12,True,True,9,108,[DeprecationWarning: Setting the shape on a Nu...,[]


## 7. Pooled MAE, RMSE, and Seasonal MASE Evaluation
Metrics are pooled over row-level out-of-sample errors; pooled RMSE is not the mean of fold RMSE values.

In [23]:
comparison_columns = ['candidate_id','role','family','eligible','pooled_mae','pooled_rmse','pooled_seasonal_mase','fold_mae_std','long_horizon_mae_h7_h12','delta_mae_vs_seasonal_naive','relative_mae_improvement_pct']
comparison = summary_table.reindex(columns=comparison_columns).sort_values(['eligible','pooled_mae'], ascending=[False,True])
comparison


,candidate_id,role,family,eligible,pooled_mae,pooled_rmse,pooled_seasonal_mase,fold_mae_std,long_horizon_mae_h7_h12,delta_mae_vs_seasonal_naive,relative_mae_improvement_pct
2,seasonal_trend_ols,candidate,DeterministicSeasonalTrendOLS,True,1.839438,2.320801,0.656960,0.388753,1.992052,-0.798525,30.270533
3,holt_winters_additive_no_trend,candidate,ExponentialSmoothing,True,1.850122,2.338465,0.660751,0.357276,1.979018,-0.787841,29.865510
9,sarima_100_011_12,candidate,SARIMAX,True,1.852289,2.305228,0.661688,0.405303,1.953598,-0.785674,29.783346
4,holt_winters_additive_damped_trend,candidate,ExponentialSmoothing,True,1.853886,2.342469,0.662140,0.380209,2.000767,-0.784077,29.722838
5,holt_winters_additive_trend,candidate,ExponentialSmoothing,True,1.857710,2.343873,0.663509,0.381700,2.001234,-0.780253,29.577869
7,autoreg_lag_1_2_12_ct,candidate,AutoReg,True,2.376971,2.993368,0.848966,0.606632,2.243636,-0.260992,9.893690
6,autoreg_lag_1_12_ct,candidate,AutoReg,True,2.607713,3.292361,0.931090,0.644302,2.556223,-0.030250,1.146731
0,seasonal_naive_12,primary_baseline,SeasonalNaive,True,2.637963,3.338898,0.941686,0.389743,2.557407,0.000000,0.000000
1,naive_last_value,secondary_baseline,NaiveLastValue,True,10.563889,13.267843,3.770738,2.181979,13.283333,7.925926,-300.456300
8,sarima_100_100_12,candidate,SARIMAX,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Horizon-Wise Forecast Error Diagnostics

In [24]:
horizon_mae = pd.DataFrame({r['candidate_id']:pd.Series(r['horizon_mae'], dtype=float) for r in summaries if r['eligible']})
horizon_mae.index.name = 'horizon'
horizon_mae


,seasonal_naive_12,naive_last_value,seasonal_trend_ols,holt_winters_additive_no_trend,holt_winters_additive_damped_trend,holt_winters_additive_trend,autoreg_lag_1_12_ct,autoreg_lag_1_2_12_ct,sarima_100_011_12
horizon,,,,,,,,,
1,3.788889,2.800000,1.975736,2.092342,2.034833,2.021285,3.400292,3.306598,2.092816
2,3.033333,2.366667,1.708126,1.709499,1.672097,1.754066,2.352615,2.424127,1.728001
3,3.477778,3.522222,2.472081,2.379151,2.364123,2.426173,3.246299,3.087823,2.672425
4,2.100000,6.566667,1.190531,1.251217,1.250443,1.236406,2.292224,2.182579,1.228642
5,2.233333,12.544444,1.269834,1.454999,1.420470,1.315639,2.236987,1.965461,1.346564
6,1.677778,19.266667,1.504632,1.440148,1.500059,1.531542,2.426796,2.095246,1.437433
7,2.055556,22.322222,2.289562,2.223131,2.264046,2.335633,2.039520,1.937264,2.270549
8,2.722222,21.822222,2.075603,1.923607,1.989427,2.084655,2.662683,2.498333,1.987541
9,2.155556,17.233333,1.559161,1.573119,1.574956,1.513305,2.069167,1.806097,1.446942


## 9. Fold Stability, Warnings, and Candidate Completeness

In [25]:
warning_failure_summary = summary_table[['candidate_id','complete','eligible','folds_completed','forecast_rows','warnings','failures']]
warning_failure_summary


,candidate_id,complete,eligible,folds_completed,forecast_rows,warnings,failures
0,seasonal_naive_12,True,True,9,108,[],[]
1,naive_last_value,True,True,9,108,[],[]
2,seasonal_trend_ols,True,True,9,108,[],[]
3,holt_winters_additive_no_trend,True,True,9,108,[],[]
4,holt_winters_additive_damped_trend,True,True,9,108,[],[]
5,holt_winters_additive_trend,True,True,9,108,[],[]
6,autoreg_lag_1_12_ct,True,True,9,108,[],[]
7,autoreg_lag_1_2_12_ct,True,True,9,108,[],[]
8,sarima_100_100_12,False,False,2,24,[DeprecationWarning: Setting the shape on a Nu...,[ForecastingModelSelectionError: Explicit opti...
9,sarima_100_011_12,True,True,9,108,[DeprecationWarning: Setting the shape on a Nu...,[]


## 10. Candidate Comparison Against Seasonal Naive
Negative delta MAE and positive relative improvement indicate improvement over the mandatory primary baseline.

In [26]:
comparison[['candidate_id','delta_mae_vs_seasonal_naive','relative_mae_improvement_pct']]


,candidate_id,delta_mae_vs_seasonal_naive,relative_mae_improvement_pct
2,seasonal_trend_ols,-0.798525,30.270533
3,holt_winters_additive_no_trend,-0.787841,29.865510
9,sarima_100_011_12,-0.785674,29.783346
4,holt_winters_additive_damped_trend,-0.784077,29.722838
5,holt_winters_additive_trend,-0.780253,29.577869
7,autoreg_lag_1_2_12_ct,-0.260992,9.893690
6,autoreg_lag_1_12_ct,-0.030250,1.146731
0,seasonal_naive_12,0.000000,0.000000
1,naive_last_value,7.925926,-300.456300
8,sarima_100_100_12,NaN,NaN


## 11. Deterministic Practical-Tie Selection
The frozen tolerance is 0.05 °F; within it, seasonal MASE, RMSE, fold stability, long-horizon MAE, complexity, and lexical ID are applied in order.

In [27]:
selection = select_winner(summaries)
selected = next(r for r in summaries if r['candidate_id'] == selection['selected_candidate_id'])
selection


{'best_raw_mae': 1.8394375020631275,
 'practical_tie_tolerance_f': 0.05,
 'finalists': ['seasonal_trend_ols',
  'holt_winters_additive_no_trend',
  'sarima_100_011_12',
  'holt_winters_additive_damped_trend',
  'holt_winters_additive_trend'],
 'tie_break_order': ['pooled_seasonal_mase',
  'pooled_rmse',
  'fold_mae_std',
  'long_horizon_mae_h7_h12',
  'complexity_rank',
  'candidate_id'],
 'selected_candidate_id': 'seasonal_trend_ols',
 'ranking': ['seasonal_trend_ols',
  'holt_winters_additive_no_trend',
  'sarima_100_011_12',
  'holt_winters_additive_damped_trend',
  'holt_winters_additive_trend',
  'autoreg_lag_1_2_12_ct',
  'autoreg_lag_1_12_ct',
  'seasonal_naive_12',
  'naive_last_value']}

## 12. Scientific Selection Analysis
The selection is mechanical and quantitative; no post-hoc candidate or hyperparameter change is allowed.

In [28]:
selection_rationale = {'winner':selected['candidate_id'],'best_raw_mae':selection['best_raw_mae'],'practical_tie_finalists':selection['finalists'],'winner_seasonal_mase':selected['pooled_seasonal_mase'],'delta_mae_vs_primary_baseline':selected['delta_mae_vs_seasonal_naive']}
selection_rationale


{'winner': 'seasonal_trend_ols',
 'best_raw_mae': 1.8394375020631275,
 'practical_tie_finalists': ['seasonal_trend_ols',
  'holt_winters_additive_no_trend',
  'sarima_100_011_12',
  'holt_winters_additive_damped_trend',
  'holt_winters_additive_trend'],
 'winner_seasonal_mase': 0.6569596546123152,
 'delta_mae_vs_primary_baseline': -0.7985254608998351}

## 13. Forecasting Model-Selection Artifact Contract
Six forecasting-specific evidence and handoff files are built; no fitted estimator is serialized.

In [29]:
artifacts = build_artifacts(project_root=PROJECT.root, preparation_handoff_path=PREPARATION_PATH, preparation_payload=preparation_payload, evidence=forecast_rows, summaries=summaries, audits=fold_audits)
list(artifacts)


['model-selection-manifest.json',
 'candidate-results.json',
 'cross-validation-results.csv',
 'validation-evidence.json',
 'selection-analysis.json',
 'model-selection-handoff.json']

## 14. Atomic Artifact Materialization

In [30]:
write_result = write_forecasting_model_selection_artifacts(project_root=PROJECT.root, artifacts=artifacts)
{'status':write_result.status,'paths':write_result.paths}


{'status': 'reused_equivalent',
 'paths': ('/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/model-selection-manifest.json',
  '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/candidate-results.json',
  '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/cross-validation-results.csv',
  '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/validation-evidence.json',
  '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/selection-analysis.json',
  '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures/artifacts/model-selection/nottem/model-selection-handoff.json')}

## 15. Fresh-Process Model-Selection Handoff Reload
The notebook performs an in-kernel fail-closed reload; the execution protocol also verifies this loader in a separate Python process.

In [31]:
reloaded = load_and_validate_forecasting_model_selection_handoff(project_root=PROJECT.root, handoff_path=MODEL_SELECTION_PATH, expected_dataset_slug='nottem')
assert reloaded['selected_candidate_id'] and reloaded['readiness']['final_model_training_ready'] is True
assert reloaded['readiness']['final_holdout_evaluated'] is False and reloaded['readiness']['final_model_trained'] is False
{'schema':reloaded['schema_version'],'selected_candidate_id':reloaded['selected_candidate_id'],'fresh_reload_ready':True}


{'schema': 'forecasting-model-selection-handoff.v1',
 'selected_candidate_id': 'seasonal_trend_ols',
 'fresh_reload_ready': True}

## 16. Notebook 04 Readiness and Final-Holdout Boundary
Notebook 04 is ready only to reconstruct the frozen winner, fit it once on full development if required, and then open/evaluate 1939 exactly once. This notebook neither trains that final model nor reads holdout values.

In [32]:
notebook_04_readiness = {'gate':'READY','selected_candidate_id':reloaded['selected_candidate_id'],'final_holdout_sealed':reloaded['readiness']['final_holdout_sealed'],'final_holdout_evaluated':reloaded['readiness']['final_holdout_evaluated'],'final_model_training_ready':reloaded['readiness']['final_model_training_ready'],'final_model_trained':reloaded['readiness']['final_model_trained']}
notebook_04_readiness


{'gate': 'READY',
 'selected_candidate_id': 'seasonal_trend_ols',
 'final_holdout_sealed': True,
 'final_holdout_evaluated': False,
 'final_model_training_ready': True,
 'final_model_trained': False}